In [ ]:
!pip install torch
!pip install scipy numpy pandas fastdtw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
RESIDUAL_SCALES = {

    'contour1': 0.05,
    'contour2': 0.04,
    'contour3':1

}

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from torch.utils.data import Dataset, DataLoader
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import random
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n DEVICE: {device}")
N_POINTS = 100
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
#please keep all weights of the model in the given directory


In [ ]:
class ModelforContour1(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(400,256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(256, 192),
        nn.BatchNorm1d(192),
        nn.ReLU(),
        nn.Dropout(0.15),



        nn.Linear(192, 128),

        nn.ReLU(),
        nn.Dropout(0.15),

        nn.Linear(128, 200),

    )


  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev.view(batch,-1),
        future.view(batch,-1)
    ],dim=1)
    residual = self.network(x)
    return residual.view(batch,N_POINTS,2)

In [ ]:
class ModelforContour2(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = nn.Sequential(
        nn.Linear(400,256),
        nn.ReLU(),
        nn.Dropout(0.15),

        nn.Linear(256,192),
        nn.ReLU(),
        nn.Dropout(0.1),

        nn.Linear(192,128),
        nn.ReLU(),
        nn.Dropout(0.15),
        )
    self.middle = nn.Sequential(
        nn.Linear(128,128),
        nn.ReLU(),
        nn.Dropout(0.06),)
    self.decoder = nn.Sequential(
        nn.Linear(128,192),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(192,256),
        nn.ReLU(),
        nn.Dropout(0.15),

        nn.Linear(256,200)

    )
  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev.view(batch,-1),
        future.view(batch,-1)
    ],dim=1)
    latent = self.encoder(x)
    latent = latent + self.middle(latent)
    residual = self.decoder(latent)
    return residual.view(batch,N_POINTS,2)

In [ ]:
class ModelforContour3(nn.Module):
  def __init__(self):
    super().__init__()
    pool_size = 1
    self.cnn_encoder = nn.Sequential(
       nn.Conv1d(2, 8, kernel_size=3, padding=1),
       nn.BatchNorm1d(8),
       nn.ReLU(),
       nn.Conv1d(8, 8, kernel_size=3, padding=1),
       nn.BatchNorm1d(8),
       nn.ReLU(),
    )
    self.adaptive_pool = nn.AdaptiveAvgPool1d(2)
    self.mlp_decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64,128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 200)
        )
  def forward(self,prev,future):
    batch = prev.shape[0]
    x = torch.cat([
        prev,
        future
    ],dim=1)
    transposed_x = x.permute(0,2,1)
    encode = self.cnn_encoder(transposed_x)
    adaptx = self.adaptive_pool(encode)
    xy = adaptx.view(batch,-1)
    residual = self.mlp_decoder(xy)
    return residual.view(batch,N_POINTS,2)

In [ ]:

def resample(pts,n=100):
  try:
    pts = np.array(pts,dtype=np.float32)
    if pts.ndim != 2:
      return None
    if pts.shape[0] == 2:
      pts = pts.T
    if pts.shape[1] != 2:
      return None
    if len(pts) < 2:
      return None
    diffs = np.diff(pts,axis=0)
    arc = np.concatenate([[0],np.cumsum(np.sqrt((diffs**2).sum(1)))])
    if arc[-1]==0:
      return None
    arc /=arc[-1]
    _,idx = np.unique(arc,return_index=True)
    arc = arc[idx]
    pts = pts[idx]
    if len(arc) < 2:
      return None
    t = np.linspace(0, 1, n)
    fx = interp1d(arc,pts[:,0],kind='linear',fill_value='extrapolate')
    fy = interp1d(arc,pts[:,1],kind='linear',fill_value='extrapolate')
    pts = np.stack([fx(t),fy(t)], axis=1)
    return pts.astype(np.float32)
  except:
    return None
def prepare_dataset_for_testing(matframe1,matframe2):
  mat1 = sio.loadmat(matframe1,squeeze_me = True,struct_as_record=False)
  mat2 = sio.loadmat(matframe2,squeeze_me = True,struct_as_record=False)
  contour1=[]
  contour2=[]
  contour3=[]
  pastcont1 = resample(mat1['annotation'][0].contour1,100)
  pastcont2 = resample(mat1['annotation'][0].contour2,100)
  pastcont3 = resample(mat1['annotation'][0].contour3,100)
  futurecont1 = resample(mat2['annotation'][0].contour1,100)
  futurecont2 = resample(mat2['annotation'][0].contour2,100)
  futurecont3 = resample(mat2['annotation'][0].contour3,100)
  contour1.append(pastcont1)
  contour1.append(futurecont1)
  contour2.append(pastcont2)
  contour2.append(futurecont2)
  contour3.append(pastcont3)
  contour3.append(futurecont3)
  return contour1,contour2,contour3



def testing_function(model_class,model_name,contour,contourname):
  model = model_class().to(DEVICE)
  model.load_state_dict(
  torch.load(
        f"/content/{model_name}.pth",
        map_location=DEVICE
    )
  )

  prev_pts = contour[0]
  future_pts = contour[1]


  all_pts = np.concatenate([
        prev_pts,
        future_pts,

    ], axis=0)

  mean = all_pts.mean(axis=0)
  std = all_pts.std(axis=0) + 1e-6
  prev_pts = (prev_pts - mean) / std
  future_pts = (future_pts - mean) / std
  linear_pts = (prev_pts+future_pts)/2
  prev_pts = torch.from_numpy(prev_pts).unsqueeze(0).to(DEVICE).float()
  future_pts = torch.from_numpy(future_pts).unsqueeze(0).to(DEVICE).float()
  linear_pts = torch.from_numpy(linear_pts).unsqueeze(0).to(DEVICE).float()
  model.eval()
  with torch.no_grad():
    prev_pts = prev_pts.to(DEVICE)
    future_pts = future_pts.to(DEVICE)
    linear_pts = linear_pts.to(DEVICE)
    pred_residual = model(prev_pts,future_pts)
    result = linear_pts + pred_residual*RESIDUAL_SCALES[contourname]
    result = result.cpu().detach().numpy()
    result = result * std + mean
  return result
def convert_to_matfile(contour1,contour2,contour3):
  data_list = [(contour1[0], contour2[0], contour3[0])]
  dtype = [('contour1', 'O'), ('contour2', 'O'), ('contour3', 'O')]
  annotation = np.array(data_list, dtype=dtype)
  # Create folder if it doesn't exist
  sio.savemat("my_contours.mat", {'annotation': annotation})
  print(f"✅ Saved to: {'my_contours.mat'}")




NameError: name 'torch' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_graphs_and_make_png(
    result_contour1,
    result_contour2,
    result_contour3,
    save_path="prediction_visualization.png"
):
    """
    Plot predicted contours separately in three subplots and save as PNG.

    Parameters
    ----------
    result_contour1 : ndarray
        Shape (100,2) or (1,100,2)

    result_contour2 : ndarray
        Shape (100,2) or (1,100,2)

    result_contour3 : ndarray
        Shape (100,2) or (1,100,2)

    save_path : str
        Output PNG filename.
    """

    # Remove batch dimension if present
    result_contour1 = np.squeeze(result_contour1)
    result_contour2 = np.squeeze(result_contour2)
    result_contour3 = np.squeeze(result_contour3)

    contours = [
        ("Contour 1", result_contour1, "green"),
        ("Contour 2", result_contour2, "orange"),
        ("Contour 3", result_contour3, "purple"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10, 4))

    for ax, (title, contour, color) in zip(axes, contours):

        # Plot contour line
        ax.plot(
            contour[:, 0],
            contour[:, 1],
            color=color,
            linewidth=2.5,
            solid_capstyle="round"
        )

        # Plot markers every 10th point
        ax.scatter(
            contour[::10, 0],
            contour[::10, 1],
            color=color,
            s=22,
            edgecolors=color,
            zorder=3
        )

        ax.set_title(title, fontsize=11)
        ax.set_aspect("equal")
        ax.invert_yaxis()

        ax.grid(True, alpha=0.3)

        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"✅ Visualization saved as {save_path}")

In [ ]:
matframe1 = "/content/Test_files/prev.mat"
matframe2 = "/content/Test_files/next.mat"
contour1,contour2,contour3 = prepare_dataset_for_testing(matframe1,matframe2)
result_contour1 = testing_function(ModelforContour1,"RI_contour1_model",contour1,"contour1")
result_contour2 = testing_function(ModelforContour2,"RI_contour2_model",contour2,"contour2")
result_contour3 = testing_function(ModelforContour3,"RI_contour3_model",contour3,"contour3")
convert_to_matfile(result_contour1,result_contour2,result_contour3)
plot_graphs_and_make_png(result_contour1,result_contour2,result_contour3,save_path="/content/predicted_contours_all.png",
    )